# Splitting data in train, validation, test

### Import libraries and tidied data

In [6]:
import scipy.sparse as sparse
import numpy as np
from pathlib import Path

data_dir = Path("data/tidied_360k")

sparse_matrix = sparse.load_npz(data_dir / "user_artist_matrix.npz")
users = np.load(data_dir / "user_sha1_mappings.npy", allow_pickle=True)
artists = np.load(data_dir / "artist_mbid_mappings.npy", allow_pickle=True)
artist_labels = np.load(data_dir / "artist_mappings.npy", allow_pickle=True)

In [16]:
type(users)

numpy.ndarray

In [18]:
sparse_matrix.shape

(19990, 19969)

In [8]:
print(artists[3])

0006c824-595a-45af-9374-238ce585fa3a


In [9]:
print(users[0])

0005937bb7e8b1992d94b493519e216317f71685


In [20]:
from scipy.sparse import csr_matrix

In [21]:
type(sparse_matrix) is csr_matrix

True

In [22]:
sparse_matrix.shape[0]

19990

In [10]:
print(sparse_matrix)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 887750 stored elements and shape (19990, 19969)>
  Coords	Values
  (0, 54)	396
  (0, 72)	34
  (0, 93)	42
  (0, 263)	83
  (0, 284)	16
  (0, 368)	515
  (0, 400)	30
  (0, 2097)	35
  (0, 2897)	23
  (0, 2981)	21
  (0, 3864)	42
  (0, 5609)	21
  (0, 5708)	14
  (0, 6072)	29
  (0, 7895)	45
  (0, 7975)	26
  (0, 9060)	70
  (0, 9320)	12
  (0, 9560)	24
  (0, 10246)	74
  (0, 10352)	90
  (0, 10614)	84
  (0, 11211)	35
  (0, 12651)	546
  (0, 14004)	552
  :	:
  (19989, 9348)	132
  (19989, 9494)	48
  (19989, 9943)	193
  (19989, 11026)	2912
  (19989, 11187)	470
  (19989, 11249)	67
  (19989, 12649)	178
  (19989, 12942)	246
  (19989, 13056)	32
  (19989, 13136)	50
  (19989, 13553)	470
  (19989, 13611)	49
  (19989, 13992)	57
  (19989, 14389)	277
  (19989, 14819)	422
  (19989, 15373)	23
  (19989, 15901)	497
  (19989, 16072)	24
  (19989, 16077)	150
  (19989, 18327)	63
  (19989, 18618)	25
  (19989, 18836)	58
  (19989, 19485)	96
  (19989, 19610)	23
  (19

### Decide train/test/validation split

In [11]:
splits = {
    "train": 0.7,
    "validation": 0.2,
    "test": 0.1
}

### Split within a user's data

This way the same user's data will be available to test on

In [12]:
rng = np.random.default_rng(1235)

sparse_matrix.sum_duplicates() # eliminate duplicates by adding them together
split_labels = np.zeros(sparse_matrix.nnz, dtype=int)

for u in range(sparse_matrix.shape[0]): # iterate through each user in the sparse matrix
    # in compressed matrix data, finds all the non-zero data corresponding to this row
    start, end = sparse_matrix.indptr[u], sparse_matrix.indptr[u + 1]
    # number of artists in the data
    num_artists = end - start
    # random order
    order = rng.permutation(num_artists)
    # store the number of artists in each split
    num_artists_validation = int(splits["validation"] * num_artists)
    num_artists_test = int(splits["test"] * num_artists)
    # assign labels for validation and test (train is zero by default from np.zeros)
    for i in range(num_artists_validation):
        split_labels[start + order[i]] = 1
    for i in range(num_artists_test):
        split_labels[start + order[num_artists_validation + i]] = 2

def isolate_split(matrix, labels, label):
    # copy the sparse matrix
    split = matrix.copy()
    split.data[labels != label] = 0 # ignore cells from other splits
    split.eliminate_zeros() # remove those cells
    return split

train = isolate_split(sparse_matrix, split_labels, 0)
validation = isolate_split(sparse_matrix, split_labels, 1)
test = isolate_split(sparse_matrix, split_labels, 2)

In [13]:
rows, cols, values = sparse.find(sparse_matrix)
max_idx = np.argmax(values)

best_row = rows[max_idx]
best_col = cols[max_idx]
max_value = values[max_idx]

In [14]:
print(validation)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 169500 stored elements and shape (19990, 19969)>
  Coords	Values
  (0, 54)	396
  (0, 284)	16
  (0, 6072)	29
  (0, 10246)	74
  (0, 10614)	84
  (0, 14408)	58
  (0, 19156)	297
  (1, 465)	293
  (1, 3860)	339
  (1, 8059)	490
  (1, 14251)	91
  (1, 17955)	99
  (1, 18306)	155
  (2, 2899)	79
  (2, 3636)	499
  (2, 10113)	123
  (2, 10790)	721
  (2, 15959)	146
  (2, 17507)	72
  (2, 18836)	528
  (2, 19438)	52
  (3, 993)	17
  (3, 4264)	25
  (3, 5471)	34
  (3, 6128)	17
  :	:
  (19986, 16363)	450
  (19986, 19531)	135
  (19987, 1441)	161
  (19987, 7330)	13
  (19987, 8696)	7
  (19987, 10580)	15
  (19987, 16144)	10
  (19987, 18492)	10
  (19987, 18821)	14
  (19988, 1379)	28
  (19988, 2773)	30
  (19988, 2993)	69
  (19988, 3846)	34
  (19988, 4361)	35
  (19988, 8968)	48
  (19988, 10432)	47
  (19989, 598)	88
  (19989, 3636)	289
  (19989, 3694)	449
  (19989, 7718)	24
  (19989, 7735)	155
  (19989, 8175)	27
  (19989, 8413)	749
  (19989, 13611)	49
  (199

In [15]:
(train.getnnz(axis=0) == 0).sum()

np.int64(3)